### 1. 패키지 설치 + 환경변수 로드

In [1]:
%pip install -qU langchain langchain_openai langgraph tavily

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 2. 그래프 준비

02번과 완전히 동일한 그래프를 다시 구성함. 이번 노트북에서 바꾸는 것은 `thread_id` 하나뿐임

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from tavily import TavilyClient

client = TavilyClient()

@tool
def search_web(query: str) -> list[dict]:
    """웹에서 최신 정보나 외부 자료를 검색합니다"""

    response = client.search(
        query=query,
        search_depth="basic",
        max_results=5,
        include_raw_content=False
    )

    return response.get("results", [])

class State(TypedDict):
    messages: Annotated[list, add_messages]

tools = [search_web]

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    answer = llm_with_tools.invoke(state["messages"])
    return {"messages": [answer]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)   # 하나의 그래프 인스턴스를 여러 세션이 공유함

In [4]:
from langchain_core.runnables import RunnableConfig

def ask(question: str, config: RunnableConfig):   # 반복 호출을 줄이기 위한 헬퍼
    for event in graph.stream({"messages": [("user", question)]}, config):
        for key, value in event.items():
            print(f"[{key}] {value['messages'][-1].content}")

config_1: RunnableConfig = {"configurable": {"thread_id": "1"}}
config_2: RunnableConfig = {"configurable": {"thread_id": "2"}}   # 같은 그래프, 다른 세션

### 3. thread "1" — 첫 대화

In [5]:
ask("내 이름은 길동이야. 기억해줘", config_1)

[chatbot] 안녕하세요, 길동님! 기억하겠습니다. 어떻게 도와드릴까요?


### 4. thread "1" — 이어가기

In [6]:
ask("내 이름이 뭐라고 했지?", config_1)   # 같은 thread_id → 앞의 대화를 복원해 기억함

[chatbot] 길동님이라고 말씀하셨습니다.


### 5. thread "2" — 같은 질문

그래프도 체크포인터도 동일하고 `thread_id`만 다름

In [7]:
ask("내 이름이 뭐라고 했지?", config_2)   # 다른 thread_id → thread "1"의 대화를 보지 못함

[chatbot] 죄송하지만, 당신의 이름은 알 수 없습니다. 대화 중에 이름을 언급하지 않으셨기 때문입니다. 원하시는 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!


### 6. thread "1" — 복귀

thread "2"에서 오간 대화가 thread "1"을 오염시키지 않았는지 확인함

In [8]:
ask("내 이름이 뭐라고 했지?", config_1)   # 여전히 기억함

[chatbot] 길동님이라고 했습니다.


### 7. 두 thread의 State 비교

In [9]:
for name, config in (("thread 1", config_1), ("thread 2", config_2)):
    snapshot = graph.get_state(config)   # thread 단위로 분리 저장된 State를 각각 조회
    messages = snapshot.values["messages"]

    print(f"\n{'=' * 50}")
    print(f"{name} — 메시지 {len(messages)}개")
    print(f"{'=' * 50}")
    for message in messages:
        print(f"{message.type}: {message.content}")


thread 1 — 메시지 6개
human: 내 이름은 길동이야. 기억해줘
ai: 안녕하세요, 길동님! 기억하겠습니다. 어떻게 도와드릴까요?
human: 내 이름이 뭐라고 했지?
ai: 길동님이라고 말씀하셨습니다.
human: 내 이름이 뭐라고 했지?
ai: 길동님이라고 했습니다.

thread 2 — 메시지 2개
human: 내 이름이 뭐라고 했지?
ai: 죄송하지만, 당신의 이름은 알 수 없습니다. 대화 중에 이름을 언급하지 않으셨기 때문입니다. 원하시는 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!


### 8. 정리

- `thread_id` 는 대화 세션을 구분하는 식별자이며, 체크포인터는 State를 thread 단위로 **분리 저장**함
- 그래프 인스턴스와 체크포인터가 같아도 `thread_id` 가 다르면 서로의 대화를 보지 못함
- 따라서 서버 한 대가 하나의 그래프로 여러 사용자의 대화를 동시에 처리할 수 있음
- 참고: [Add memory](https://docs.langchain.com/oss/python/langgraph/add-memory) — "Each invocation with a different `thread_id` maintains separate conversation state"